# SIREN A3 — Laiba Noor (25I-8035)
## Improvement IV: Helmholtz k-Ablation + Wave Equation IC Variation
## Bonus: Medical Image Super-Resolution (IXI Brain MRI)

**Group ID09 | FAST-NUCES | Dr. Zohair Ahmed | April 2026**

---
| Section | Experiments | Steps | Est. Time |
|---|---|---|---|
| **IV-A** | Helmholtz k ∈ {1, 5, 20, 50} | 500 each | ~15 min |
| **IV-B** | Wave IC: default/Gaussian/Double-sin | 500 each | ~15 min |
| **Bonus** | MRI SR: Bicubic/ReLU/SIREN/SIREN+FFL+CAOS | 400 each | ~20 min |
| **Plots** | 8 figures for A3 report | — | — |

**A2 Baselines (from report):**  
- Helmholtz k=1: 5,581,349 → 175,564 (96.9%, 5K steps)
- Wave IC-1: 85,955,296 → 16,236 (99.98%, 5K steps)


---
## CELL 1 — Setup


In [6]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_ROOT = '/content/drive/MyDrive/SIREN_A3_Laiba'
os.makedirs(DRIVE_ROOT, exist_ok=True)

if not os.path.exists('/content/siren'):
    !git clone https://github.com/vsitzmann/siren.git /content/siren -q
    !sed -i 's/NeuralProcessImplicit2DHypernetBVP/NeuralProcessImplicit2DHypernet/g' /content/siren/utils.py
    !sed -i 's/from skimage.measure import compare_psnr, compare_ssim/from skimage.metrics import peak_signal_noise_ratio as compare_psnr, structural_similarity as compare_ssim/g' /content/siren/utils.py

!pip install configargparse cmapy scikit-video scikit-image torchio -q

import sys, time, csv
sys.path.insert(0, '/content/siren')
import torch, numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch.utils.data import Dataset
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
GPU: Tesla T4 | VRAM: 15.6 GB


## CELL 2 — Model Definitions


In [7]:
class SineLayer(nn.Module):
    def __init__(self, in_f, out_f, is_first=False, omega_0=30.):
        super().__init__()
        self.omega_0 = omega_0
        self.linear  = nn.Linear(in_f, out_f)
        with torch.no_grad():
            if is_first:
                self.linear.weight.uniform_(-1/in_f, 1/in_f)
            else:
                self.linear.weight.uniform_(-np.sqrt(6/in_f)/omega_0,
                                             np.sqrt(6/in_f)/omega_0)
    def forward(self, x):
        return torch.sin(self.omega_0 * self.linear(x))

class Siren(nn.Module):
    def __init__(self, in_f=2, hid=256, layers=4, out_f=1, omega_0=30.):
        super().__init__()
        self.net = nn.Sequential(
            SineLayer(in_f, hid, is_first=True, omega_0=omega_0),
            *[SineLayer(hid, hid, omega_0=omega_0) for _ in range(layers-1)],
            nn.Linear(hid, out_f)
        )
        with torch.no_grad():
            self.net[-1].weight.uniform_(-np.sqrt(6/hid)/omega_0,
                                          np.sqrt(6/hid)/omega_0)
    def forward(self, x): return self.net(x)

class ReLUMLP(nn.Module):
    def __init__(self, in_f=2, hid=256, layers=4, out_f=1):
        super().__init__()
        ls = [nn.Linear(in_f, hid), nn.ReLU()]
        for _ in range(layers-1): ls += [nn.Linear(hid,hid), nn.ReLU()]
        ls.append(nn.Linear(hid, out_f))
        self.net = nn.Sequential(*ls)
    def forward(self, x): return self.net(x)

print('SineLayer, Siren, ReLUMLP defined.')

SineLayer, Siren, ReLUMLP defined.


---
# IMPROVEMENT IV-A: Helmholtz Equation — k-Ablation
∇²u + k²n²u = −q  |  sidelength=150, **1000 steps per k**, 4 runs total


## CELL 3 — Helmholtz Training (4 k values × 500 steps)


In [8]:
K_VALUES  = [1, 5, 20, 50]
H_STEPS   = 1000
H_LR      = 2e-5
H_SIDE    = 150
LOG_EVERY = 50
A2_H_INIT, A2_H_FINAL = 5_581_349, 175_564
COLORS = ['#1f77b4','#ff7f0e','#2ca02c','#d62728']

def build_helm_batch(side, k):
    lin = torch.linspace(-1, 1, side)
    x, y = torch.meshgrid(lin, lin, indexing='ij')
    coords = torch.stack([x.reshape(-1), y.reshape(-1)], dim=-1)
    dists  = torch.norm(coords, dim=-1)
    source = torch.exp(-dists**2/(2*0.01**2)).unsqueeze(-1)
    vel    = torch.ones(coords.shape[0], 1)
    return coords, source, vel

def train_helmholtz(k):
    print(f'  k={k} ...')
    c_cpu, src, vel = build_helm_batch(H_SIDE, k)
    src, vel = src.to(device), vel.to(device)
    model = Siren(in_f=2, hid=256, layers=4, out_f=2).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=H_LR)
    losses = []
    t0 = time.time()
    for step in range(H_STEPS):
        model.train()
        c = c_cpu.to(device).requires_grad_(True)
        u = model(c)
        laps = []
        for ch in range(2):
            g = torch.autograd.grad(u[:,ch:ch+1], c,
                    grad_outputs=torch.ones_like(u[:,ch:ch+1]),
                    create_graph=True)[0]
            lap = sum(torch.autograd.grad(g[:,d:d+1], c,
                        grad_outputs=torch.ones_like(g[:,d:d+1]),
                        create_graph=True)[0][:,d:d+1] for d in range(2))
            laps.append(lap)
        k2n2 = k**2 * vel**2
        loss  = ((laps[0]+k2n2*u[:,0:1]+src)**2).sum() + ((laps[1]+k2n2*u[:,1:2])**2).sum()
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
        if step % LOG_EVERY == 0: print(f'    {step:4d} | {losses[-1]:>16,.0f}')
    elapsed = (time.time()-t0)/60
    model.eval()
    with torch.no_grad():
        wf = model(c_cpu.to(device)).cpu().numpy()
    red = (1-losses[-1]/losses[0])*100
    print(f'  Done {elapsed:.1f}m | {losses[0]:.0f}→{losses[-1]:.0f} ({red:.1f}%)')
    return {'losses':losses,'init':losses[0],'final':losses[-1],
            'reduction':red,'time_min':elapsed,'wf':wf,'side':H_SIDE}

print('=== Helmholtz k-Ablation (1000 steps × 4 runs) ===')
helm = {k: train_helmholtz(k) for k in K_VALUES}
print('Done!')

=== Helmholtz k-Ablation (1000 steps × 4 runs) ===
  k=1 ...
       0 |        8,543,346
      50 |           25,509
     100 |            7,739
     150 |            4,522
     200 |            3,032
     250 |            2,203
     300 |            1,690
     350 |            1,346
     400 |            1,104
     450 |              926
     500 |              790
     550 |              683
     600 |              597
     650 |              527
     700 |              469
     750 |              420
     800 |              379
     850 |              343
     900 |              312
     950 |              284
  Done 3.8m | 8543346→261 (100.0%)
  k=5 ...
       0 |        6,551,798
      50 |           18,073
     100 |            5,317
     150 |            3,078
     200 |            2,046
     250 |            1,473
     300 |            1,120
     350 |              886
     400 |              721
     450 |              601
     500 |              510
     550 |              43

## CELL 4 — Figure 1: Loss Curves + Final Loss + Reduction %


In [9]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

ax = axes[0]
for i,k in enumerate(K_VALUES):
    ax.semilogy(helm[k]['losses'], color=COLORS[i], label=f'k={k}', lw=2.2)
ax.axhline(A2_H_FINAL, color='gray', ls='--', lw=1.5, label='A2 final (5K steps)')
ax.set_xlabel('Step',fontsize=13); ax.set_ylabel('Loss (log)',fontsize=13)
ax.set_title('Helmholtz Loss Curves — k-Ablation',fontsize=14,fontweight='bold')
ax.legend(fontsize=11); ax.grid(True,alpha=0.3)

ax2 = axes[1]
finals = [helm[k]['final'] for k in K_VALUES]
bars = ax2.bar([f'k={k}' for k in K_VALUES], finals, color=COLORS, edgecolor='black', lw=0.8)
for b,v in zip(bars,finals):
    ax2.text(b.get_x()+b.get_width()/2, b.get_height()*1.03, f'{v:.0f}',
             ha='center',va='bottom',fontsize=10,fontweight='bold')
ax2.set_ylabel('Final Loss',fontsize=13)
ax2.set_title('Final Loss vs k',fontsize=14,fontweight='bold'); ax2.grid(True,alpha=0.3,axis='y')

ax3 = axes[2]
reds = [helm[k]['reduction'] for k in K_VALUES]
bars3 = ax3.bar([f'k={k}' for k in K_VALUES], reds, color=COLORS, edgecolor='black', lw=0.8)
for b,v in zip(bars3,reds):
    ax3.text(b.get_x()+b.get_width()/2, b.get_height()+0.5, f'{v:.1f}%',
             ha='center',va='bottom',fontsize=11,fontweight='bold')
ax3.set_ylabel('Loss Reduction (%)',fontsize=13); ax3.set_ylim(0,108)
ax3.set_title('Reduction % vs k',fontsize=14,fontweight='bold'); ax3.grid(True,alpha=0.3,axis='y')

plt.suptitle('FIGURE 1 — Helmholtz k-Ablation: Loss Curves / Final Loss / Reduction',
             fontsize=14,fontweight='bold',y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_ROOT,'fig1_helmholtz_loss.png'),dpi=150,bbox_inches='tight')
plt.show(); print('Saved fig1')

Saved fig1


## CELL 5 — Figure 2: Helmholtz Wavefields (Real / Imaginary / Amplitude)


In [10]:
S = H_SIDE
fig, axes = plt.subplots(3, 4, figsize=(22, 15))
row_labels = ['Real part  u_r', 'Imaginary part  u_i', 'Amplitude  |u|']

for col,k in enumerate(K_VALUES):
    wf = helm[k]['wf']
    r, im_wf = wf[:,0].reshape(S,S), wf[:,1].reshape(S,S)
    amp = np.sqrt(r**2 + im_wf**2)
    vr = np.percentile(np.abs(r),99) or 1
    vi = np.percentile(np.abs(im_wf),99) or 1

    i0 = axes[0,col].imshow(r,  cmap='seismic',origin='lower',vmin=-vr,vmax=vr)
    axes[0,col].set_title(f'k={k}\n{row_labels[0]}',fontsize=11,fontweight='bold')
    axes[0,col].axis('off'); plt.colorbar(i0,ax=axes[0,col],fraction=0.046)

    i1 = axes[1,col].imshow(im_wf,cmap='seismic',origin='lower',vmin=-vi,vmax=vi)
    axes[1,col].set_title(f'k={k}\n{row_labels[1]}',fontsize=11,fontweight='bold')
    axes[1,col].axis('off'); plt.colorbar(i1,ax=axes[1,col],fraction=0.046)

    i2 = axes[2,col].imshow(amp, cmap='inferno',origin='lower')
    axes[2,col].set_title(f'k={k}\n{row_labels[2]}',fontsize=11,fontweight='bold')
    axes[2,col].axis('off'); plt.colorbar(i2,ax=axes[2,col],fraction=0.046)

plt.suptitle('FIGURE 2 — Helmholtz Wavefield: Real / Imaginary / Amplitude per k',
             fontsize=14,fontweight='bold',y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_ROOT,'fig2_helmholtz_wavefields.png'),dpi=150,bbox_inches='tight')
plt.show(); print('Saved fig2')

Saved fig2


## CELL 6 — Figure 3: Convergence Speed + Helmholtz Table


In [11]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
for i,k in enumerate(K_VALUES):
    ls = np.array(helm[k]['losses'])
    ax.plot(ls/ls[0], color=COLORS[i], label=f'k={k}', lw=2.2)
ax.axhline(0.05, color='black', ls=':', lw=1.5, label='95% reduction')
ax.set_xlabel('Step',fontsize=13); ax.set_ylabel('Normalised Loss',fontsize=13)
ax.set_title('Convergence Speed — Normalised Loss',fontsize=14,fontweight='bold')
ax.legend(fontsize=11); ax.grid(True,alpha=0.3)

ax2 = axes[1]
steps95 = []
for k in K_VALUES:
    ls = np.array(helm[k]['losses']); norm = ls/ls[0]
    idx = np.where(norm <= 0.05)[0]
    steps95.append(int(idx[0]) if len(idx) else H_STEPS)
bars = ax2.bar([f'k={k}' for k in K_VALUES], steps95, color=COLORS, edgecolor='black', lw=0.8)
for b,v in zip(bars,steps95):
    label = str(v) if v < H_STEPS else f'>{H_STEPS}'
    ax2.text(b.get_x()+b.get_width()/2, b.get_height()+3, label,
             ha='center',va='bottom',fontsize=11,fontweight='bold')
ax2.set_ylabel('Steps to 95% Reduction',fontsize=13)
ax2.set_title('Convergence Speed per k',fontsize=14,fontweight='bold'); ax2.grid(True,alpha=0.3,axis='y')

plt.suptitle('FIGURE 3 — Helmholtz Convergence Speed Analysis',
             fontsize=14,fontweight='bold',y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_ROOT,'fig3_helmholtz_convergence.png'),dpi=150,bbox_inches='tight')
plt.show(); print('Saved fig3')

print('\n'+'-'*80)
print(f'{"Table 1 — Helmholtz k-Ablation":^80}')
print('-'*80)
print(f'{"Config":<25} {"k":>4} {"Init Loss":>14} {"Final Loss":>12} {"Reduction":>10} {"Steps":>6}')
print('─'*80)
print(f'{"SIREN A2 baseline":<25} {1:>4} {A2_H_INIT:>14,.0f} {A2_H_FINAL:>12,.0f} {"96.9%":>10} {"5000":>6}')
print('─'*80)
for k in K_VALUES:
    r=helm[k]; tag='repro' if k==1 else 'new'
    print(f'{f"SIREN k={k} [{tag}]":<25} {k:>4} {r["init"]:>14,.0f} {r["final"]:>12,.0f} {r["reduction"]:>9.1f}% {H_STEPS:>6}')
print('-'*80)

Saved fig3

--------------------------------------------------------------------------------
                         Table 1 — Helmholtz k-Ablation                         
--------------------------------------------------------------------------------
Config                       k      Init Loss   Final Loss  Reduction  Steps
────────────────────────────────────────────────────────────────────────────────
SIREN A2 baseline            1      5,581,349      175,564      96.9%   5000
────────────────────────────────────────────────────────────────────────────────
SIREN k=1 [repro]            1      8,543,346          261     100.0%   1000
SIREN k=5 [new]              5      6,551,798          166     100.0%   1000
SIREN k=20 [new]            20     17,054,916          586     100.0%   1000
SIREN k=50 [new]            50    107,369,632       24,346     100.0%   1000
--------------------------------------------------------------------------------


---
# IMPROVEMENT IV-B: Wave Equation — Initial Condition Variation
∂²u/∂t² = c²(∂²u/∂x²+∂²u/∂y²)  |  3D input (x,y,t)  |  **500 steps per IC**

| IC | Formula |
|---|---|
| IC-1 (A2) | sin(πx) |
| IC-2 NEW | exp(−‖x‖²/0.05) — Gaussian |
| IC-3 NEW | sin(2πx)+0.5·sin(4πx) — Double-sin |


## CELL 7 — Wave Training (3 ICs × 500 steps)


In [12]:
W_STEPS  = 500
W_LR     = 2e-5
W_BATCH  = 3000
A2_W_INIT, A2_W_FINAL = 85_955_296, 16_236

IC_TYPES  = ['default','gaussian','double_sin']
IC_LABELS = {'default':'IC-1: sin(πx) [A2]','gaussian':'IC-2: Gaussian [NEW]','double_sin':'IC-3: Double-sin [NEW]'}
IC_COLORS = ['#1f77b4','#2ca02c','#d62728']

def ic_val(ic, x, y):
    if ic=='default':    return torch.sin(np.pi*x)
    elif ic=='gaussian': return torch.exp(-(x**2+y**2)/0.05)
    else:                return torch.sin(2*np.pi*x)+0.5*torch.sin(4*np.pi*x)

def train_wave(ic):
    print(f'  IC={ic} ...')
    model = Siren(in_f=3, hid=256, layers=3, out_f=1).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=W_LR)
    losses = []
    t0 = time.time()
    for step in range(W_STEPS):
        model.train()
        coords = (torch.rand(W_BATCH,3)*2-1).to(device).requires_grad_(True)
        u = model(coords)
        du = torch.autograd.grad(u,coords,grad_outputs=torch.ones_like(u),create_graph=True)[0]
        d2t = torch.autograd.grad(du[:,2:3],coords,grad_outputs=torch.ones_like(du[:,2:3]),create_graph=True)[0][:,2:3]
        d2x = torch.autograd.grad(du[:,0:1],coords,grad_outputs=torch.ones_like(du[:,0:1]),create_graph=True)[0][:,0:1]
        d2y = torch.autograd.grad(du[:,1:2],coords,grad_outputs=torch.ones_like(du[:,1:2]),create_graph=True)[0][:,1:2]
        pde_loss = ((d2t-d2x-d2y)**2).sum()

        n_ic = W_BATCH//5
        xy_ic = (torch.rand(n_ic,2)*2-1).to(device)
        c_ic  = torch.cat([xy_ic, torch.zeros(n_ic,1).to(device)],dim=-1).requires_grad_(True)
        u_ic  = model(c_ic)
        gt_ic = ic_val(ic, xy_ic[:,0], xy_ic[:,1]).unsqueeze(-1).to(device)
        ic_loss = ((u_ic-gt_ic)**2).sum()
        du_ic   = torch.autograd.grad(u_ic,c_ic,grad_outputs=torch.ones_like(u_ic),create_graph=True)[0]
        vel_loss= (du_ic[:,2:3]**2).sum()

        loss = pde_loss + ic_loss + vel_loss
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
        if step % LOG_EVERY==0: print(f'    {step:4d} | {losses[-1]:>18,.0f}')

    elapsed=(time.time()-t0)/60
    red=(1-losses[-1]/losses[0])*100
    print(f'  Done {elapsed:.1f}m | {losses[0]:.0f}→{losses[-1]:.0f} ({red:.1f}%)')

    # Build spacetime visualisation (y=0 slice)
    lin=torch.linspace(-1,1,80); tv=torch.linspace(0,1,80)
    gx,gt=torch.meshgrid(lin,tv,indexing='ij')
    vc=torch.stack([gx.reshape(-1),torch.zeros_like(gx).reshape(-1),gt.reshape(-1)],dim=-1)
    model.eval()
    with torch.no_grad(): u_vis=model(vc.to(device)).cpu().numpy().reshape(80,80)

    return {'losses':losses,'init':losses[0],'final':losses[-1],'reduction':red,'time_min':elapsed,'u_vis':u_vis}

print('=== Wave IC Variation (500 steps × 3 ICs) ===')
wave = {ic: train_wave(ic) for ic in IC_TYPES}
print('Done!')

=== Wave IC Variation (500 steps × 3 ICs) ===
  IC=default ...
       0 |             71,598
      50 |              4,088
     100 |              1,715
     150 |              1,099
     200 |                835
     250 |                684
     300 |                603
     350 |                530
     400 |                495
     450 |                464
  Done 0.2m | 71598→423 (99.4%)
  IC=gaussian ...
       0 |             56,604
      50 |              2,892
     100 |              1,075
     150 |                645
     200 |                426
     250 |                331
     300 |                241
     350 |                200
     400 |                161
     450 |                133
  Done 0.2m | 56604→116 (99.8%)
  IC=double_sin ...
       0 |             70,266
      50 |              4,040
     100 |              1,751
     150 |              1,162
     200 |                909
     250 |                769
     300 |                675
     350 |               

## CELL 8 — Figure 4: Wave Loss Curves / Final Loss / Normalised Convergence


In [13]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

ax=axes[0]
for i,ic in enumerate(IC_TYPES):
    ax.semilogy(wave[ic]['losses'],color=IC_COLORS[i],label=IC_LABELS[ic],lw=2.2)
ax.axhline(A2_W_FINAL,color='gray',ls='--',lw=1.5,label='A2 final (5K steps)')
ax.set_xlabel('Step',fontsize=13); ax.set_ylabel('Loss (log)',fontsize=13)
ax.set_title('Wave Equation Loss — IC Variation',fontsize=14,fontweight='bold')
ax.legend(fontsize=10); ax.grid(True,alpha=0.3)

ax2=axes[1]
wfin=[wave[ic]['final'] for ic in IC_TYPES]
short=['IC-1\nsin(πx)','IC-2\nGaussian','IC-3\nDouble-sin']
bars=ax2.bar(short,wfin,color=IC_COLORS,edgecolor='black',lw=0.8)
for b,v in zip(bars,wfin):
    ax2.text(b.get_x()+b.get_width()/2,b.get_height()*1.03,f'{v:.0f}',
             ha='center',va='bottom',fontsize=10,fontweight='bold')
ax2.set_ylabel('Final Loss at 500 Steps',fontsize=13)
ax2.set_title('Final Loss by IC',fontsize=14,fontweight='bold'); ax2.grid(True,alpha=0.3,axis='y')

ax3=axes[2]
for i,ic in enumerate(IC_TYPES):
    ls=np.array(wave[ic]['losses'])
    ax3.plot(ls/ls[0],color=IC_COLORS[i],label=IC_LABELS[ic],lw=2.2)
ax3.axhline(0.05,color='black',ls=':',lw=1.5,label='95% reduction')
ax3.set_xlabel('Step',fontsize=13); ax3.set_ylabel('Normalised Loss',fontsize=13)
ax3.set_title('Convergence Speed — Normalised',fontsize=14,fontweight='bold')
ax3.legend(fontsize=10); ax3.grid(True,alpha=0.3)

plt.suptitle('FIGURE 4 — Wave Equation IC Variation (Improvement IV-B)',
             fontsize=14,fontweight='bold',y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_ROOT,'fig4_wave_loss.png'),dpi=150,bbox_inches='tight')
plt.show(); print('Saved fig4')

Saved fig4


## CELL 9 — Figure 5: Wave Spacetime Solutions u(x,t)


In [14]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

for col,ic in enumerate(IC_TYPES):
    uv=wave[ic]['u_vis']
    vm=np.percentile(np.abs(uv),99) or 1

    im1=axes[0,col].imshow(uv,cmap='RdBu',origin='lower',vmin=-vm,vmax=vm,
                            aspect='auto',extent=[-1,1,0,1])
    axes[0,col].set_title(f'{IC_LABELS[ic]}\nu(x,t)',fontsize=11,fontweight='bold')
    axes[0,col].set_xlabel('x'); axes[0,col].set_ylabel('t')
    plt.colorbar(im1,ax=axes[0,col],fraction=0.046)

    im2=axes[1,col].imshow(np.abs(uv),cmap='hot',origin='lower',
                            aspect='auto',extent=[-1,1,0,1])
    axes[1,col].set_title(f'{IC_LABELS[ic]}\n|u(x,t)| Amplitude',fontsize=11,fontweight='bold')
    axes[1,col].set_xlabel('x'); axes[1,col].set_ylabel('t')
    plt.colorbar(im2,ax=axes[1,col],fraction=0.046)

plt.suptitle('FIGURE 5 — Wave Spacetime Solutions u(x,t) per Initial Condition',
             fontsize=14,fontweight='bold',y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_ROOT,'fig5_wave_spacetime.png'),dpi=150,bbox_inches='tight')
plt.show(); print('Saved fig5')

Saved fig5


## CELL 10 — Figure 6: IC Profiles + Wave Table


In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
xv=torch.linspace(-1,1,300); yv=torch.zeros(300)
formulas={'default':'sin(πx)','gaussian':'exp(−‖x‖²/0.05)','double_sin':'sin(2πx)+0.5·sin(4πx)'}

for col,ic in enumerate(IC_TYPES):
    vals=ic_val(ic,xv,yv).numpy()
    axes[col].plot(xv.numpy(),vals,color=IC_COLORS[col],lw=2.5)
    axes[col].fill_between(xv.numpy(),0,vals,alpha=0.2,color=IC_COLORS[col])
    axes[col].set_title(f'{IC_LABELS[ic]}\nu(x,0)={formulas[ic]}',fontsize=11,fontweight='bold')
    axes[col].set_xlabel('x',fontsize=12); axes[col].set_ylabel('u(x,0)',fontsize=12)
    axes[col].axhline(0,color='gray',lw=0.8); axes[col].grid(True,alpha=0.3)

plt.suptitle('FIGURE 6 — Initial Condition Profiles u(x,y=0,t=0)',
             fontsize=14,fontweight='bold',y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_ROOT,'fig6_wave_ic_profiles.png'),dpi=150,bbox_inches='tight')
plt.show(); print('Saved fig6')

print('\n'+'-'*80)
print(f'{"Table 2 — Wave Equation IC Variation":^80}')
print('-'*80)
print(f'{"Experiment":<36} {"Init Loss":>14} {"Final Loss":>11} {"Reduction":>10} {"Steps":>6}')
print('─'*80)
print(f'{"SIREN A2 (IC-1, default)":<36} {A2_W_INIT:>14,.0f} {A2_W_FINAL:>11,.0f} {"99.98%":>10} {"5000":>6}')
print('─'*80)
for ic in IC_TYPES:
    r=wave[ic]
    print(f'{IC_LABELS[ic]:<36} {r["init"]:>14,.0f} {r["final"]:>11,.0f} {r["reduction"]:>9.1f}% {W_STEPS:>6}')
print('-'*80)

Saved fig6

--------------------------------------------------------------------------------
                      Table 2 — Wave Equation IC Variation                      
--------------------------------------------------------------------------------
Experiment                                Init Loss  Final Loss  Reduction  Steps
────────────────────────────────────────────────────────────────────────────────
SIREN A2 (IC-1, default)                 85,955,296      16,236     99.98%   5000
────────────────────────────────────────────────────────────────────────────────
IC-1: sin(πx) [A2]                           71,598         423      99.4%    500
IC-2: Gaussian [NEW]                         56,604         116      99.8%    500
IC-3: Double-sin [NEW]                       70,266         492      99.3%    500
--------------------------------------------------------------------------------


---
# BONUS: SIREN for MRI Super-Resolution (IXI Brain MRI)

**Task:** (x,y) → pixel intensity. Train on LR, evaluate PSNR/SSIM on HR.  
**5 slices × 2 scales (2×, 4×) × 4 methods = 40 runs × ~30s each**


## CELL 11 — MRI Data Setup


In [16]:
from skimage.transform import resize as sk_resize
from skimage.metrics import peak_signal_noise_ratio as calc_psnr
from skimage.metrics import structural_similarity as calc_ssim

TARGET_SZ  = 128
NUM_SLICES = 5
SCALES_MRI = [2, 4]
MRI_DIR    = '/content/ixi_mri'
os.makedirs(MRI_DIR, exist_ok=True)

hr_slices = []
try:
    import torchio as tio
    ixi = tio.datasets.IXI(root=MRI_DIR, modalities=['T1'], download=True)
    for subj in ixi:
        if len(hr_slices)>=NUM_SLICES: break
        try:
            vol = subj['T1'][tio.DATA].squeeze().numpy()
            sl  = vol[:,:,vol.shape[2]//2].astype(np.float32)
            sl  = sk_resize(sl,(TARGET_SZ,TARGET_SZ),anti_aliasing=True,preserve_range=True)
            sl  = (sl-sl.min())/(sl.max()-sl.min()+1e-8)
            hr_slices.append(sl)
        except: pass
    print(f'IXI loaded: {len(hr_slices)} slices')
except Exception as e:
    print(f'IXI failed ({e}). Using synthetic slices.')

if len(hr_slices) < NUM_SLICES:
    print('Generating synthetic brain-like slices...')
    np.random.seed(42)
    while len(hr_slices) < NUM_SLICES:
        sl = np.zeros((TARGET_SZ,TARGET_SZ),np.float32)
        y,x = np.ogrid[-1:1:TARGET_SZ*1j,-1:1:TARGET_SZ*1j]
        brain=(x**2/0.7**2+y**2/0.85**2)<1.0
        grey =(x**2/0.5**2+y**2/0.60**2)<1.0
        sl[brain]=0.3+0.08*np.random.randn(TARGET_SZ,TARGET_SZ)[brain]
        sl[grey&brain]+=0.35+0.06*np.random.randn(TARGET_SZ,TARGET_SZ)[grey&brain]
        sl=np.clip(sl+0.04*np.random.randn(TARGET_SZ,TARGET_SZ)*brain,0,1)
        hr_slices.append(sl)
    print(f'Generated {len(hr_slices)} synthetic slices')

def downsample(img,s): return sk_resize(img,(img.shape[0]//s,img.shape[1]//s),anti_aliasing=True,preserve_range=True)
def bicubic_up(lr,sz): return sk_resize(lr,(sz,sz),order=3,anti_aliasing=True,preserve_range=True)

print(f'{len(hr_slices)} HR slices ({TARGET_SZ}×{TARGET_SZ}) ready.')

100%|██████████| 4840816640/4840816640 [02:41<00:00, 30041871.64it/s]


IXI loaded: 5 slices
5 HR slices (128×128) ready.


## CELL 12 — MRI Training Functions


In [17]:
MRI_STEPS = 400
MRI_LR    = 1e-4

def make_grid(sz):
    lin=torch.linspace(-1,1,sz)
    gy,gx=torch.meshgrid(lin,lin,indexing='ij')
    return torch.stack([gx.reshape(-1),gy.reshape(-1)],dim=-1)

def caos_omega(step,total,lo=10.,hi=60.):
    return lo+(hi-lo)*(1-np.cos(np.pi*step/total))/2

def train_mri(lr_img,hr_img,mtype='siren',use_ffl=False,use_caos=False):
    H=lr_img.shape[0]; Hr=hr_img.shape[0]
    model=(Siren(in_f=2,hid=256,layers=5,out_f=1) if mtype=='siren'
           else ReLUMLP(in_f=2,hid=256,layers=5,out_f=1)).to(device)
    opt=torch.optim.Adam(model.parameters(),lr=MRI_LR)
    clr=make_grid(H).to(device)
    plr=torch.from_numpy(lr_img.reshape(-1,1)).float().to(device)
    chr_=make_grid(Hr).to(device)
    losses=[]
    for step in range(MRI_STEPS):
        model.train()
        if use_caos and mtype=='siren' and step%40==0:
            w=caos_omega(step,MRI_STEPS)
            for m in model.net.modules():
                if isinstance(m,SineLayer): m.omega_0=w
        pred=model(clr); l2=((pred-plr)**2).mean(); loss=l2
        if use_ffl:
            lam=0.05*min(1.,step/200)
            p2d=pred.reshape(H,H); g2d=plr.reshape(H,H)
            loss=l2+lam*torch.mean(torch.abs(torch.abs(torch.fft.fft2(p2d))-torch.abs(torch.fft.fft2(g2d))))
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    model.eval()
    with torch.no_grad(): ph=model(chr_).cpu().numpy().reshape(Hr,Hr)
    ph=np.clip(ph,0,1)
    return {'psnr':calc_psnr(hr_img,ph,data_range=1.),'ssim':calc_ssim(hr_img,ph,data_range=1.),
            'pred':ph,'losses':losses}

METHODS=[('bicubic','Bicubic',False,False),('relu','ReLU MLP',False,False),
         ('siren','SIREN (A2)',False,False),('siren_ffl_caos','SIREN+FFL+CAOS',True,True)]
MC=['#aec7e8','#ffbb78','#98df8a','#ff9896']
print('MRI functions ready.')

MRI functions ready.


## CELL 13 — Run MRI Experiments


In [18]:
mri_res={s:{m[0]:[] for m in METHODS} for s in SCALES_MRI}
mri_ex={}

for scale in SCALES_MRI:
    print(f'\n--- Scale {scale}× ---')
    for si,hr in enumerate(hr_slices):
        lr=downsample(hr,scale)
        print(f'  Slice {si+1} | LR={lr.shape}')
        for mkey,mname,ffl,caos in METHODS:
            if mkey=='bicubic':
                bic=bicubic_up(lr,TARGET_SZ)
                res={'psnr':calc_psnr(hr,bic,data_range=1.),'ssim':calc_ssim(hr,bic,data_range=1.),'pred':bic,'losses':[]}
            else:
                mtype='relu' if mkey=='relu' else 'siren'
                res=train_mri(lr,hr,mtype=mtype,use_ffl=ffl,use_caos=caos)
            mri_res[scale][mkey].append(res)
            print(f'    {mname:<28} PSNR={res["psnr"]:6.2f}  SSIM={res["ssim"]:.4f}')
        if si==0:
            mri_ex[(scale,'hr')]=hr; mri_ex[(scale,'lr')]=lr
            for mkey,_,_,_ in METHODS: mri_ex[(scale,mkey)]=mri_res[scale][mkey][0]['pred']

print('\n✓ MRI done!')


--- Scale 2× ---
  Slice 1 | LR=(64, 64)
    Bicubic                      PSNR= 26.78  SSIM=0.8942
    ReLU MLP                     PSNR= 19.17  SSIM=0.4220
    SIREN (A2)                   PSNR= 25.98  SSIM=0.8781
    SIREN+FFL+CAOS               PSNR= 14.37  SSIM=0.0831
  Slice 2 | LR=(64, 64)
    Bicubic                      PSNR= 26.95  SSIM=0.8792
    ReLU MLP                     PSNR= 18.97  SSIM=0.3377
    SIREN (A2)                   PSNR= 26.30  SSIM=0.8601
    SIREN+FFL+CAOS               PSNR= 14.14  SSIM=0.1154
  Slice 3 | LR=(64, 64)
    Bicubic                      PSNR= 27.88  SSIM=0.8742
    ReLU MLP                     PSNR= 20.25  SSIM=0.3310
    SIREN (A2)                   PSNR= 27.20  SSIM=0.8529
    SIREN+FFL+CAOS               PSNR= 15.66  SSIM=0.1017
  Slice 4 | LR=(64, 64)
    Bicubic                      PSNR= 26.28  SSIM=0.8800
    ReLU MLP                     PSNR= 19.09  SSIM=0.4261
    SIREN (A2)                   PSNR= 25.68  SSIM=0.8622
    SIREN+FFL+CA

## CELL 14 — Figure 7: MRI Visual Comparison


In [19]:
order=['hr','lr','bicubic','relu','siren','siren_ffl_caos']
titles={'hr':'HR Ground Truth','lr':'LR Input\n(upscaled)','bicubic':'Bicubic',
        'relu':'ReLU MLP','siren':'SIREN (A2)','siren_ffl_caos':'SIREN+FFL\n+CAOS (A3)'}

fig,axes=plt.subplots(2,6,figsize=(24,9))
for row,scale in enumerate(SCALES_MRI):
    for col,mk in enumerate(order):
        ax=axes[row,col]
        img=mri_ex.get((scale,mk))
        if img is None: ax.axis('off'); continue
        show=bicubic_up(img,TARGET_SZ) if mk=='lr' else img
        ax.imshow(show,cmap='gray',vmin=0,vmax=1)
        t=titles[mk]
        if mk not in('hr','lr'):
            d=mri_res[scale][mk][0]
            t+=f'\n{d["psnr"]:.2f} dB | {d["ssim"]:.3f}'
        ax.set_title(t,fontsize=9,fontweight='bold'); ax.axis('off')
        if col==0: ax.set_ylabel(f'{scale}× SR',fontsize=12,fontweight='bold')

plt.suptitle('FIGURE 7 — MRI Super-Resolution Visual Comparison (Slice 1, IXI T1 Brain)',
             fontsize=13,fontweight='bold',y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_ROOT,'fig7_mri_visual.png'),dpi=150,bbox_inches='tight')
plt.show(); print('Saved fig7')

Saved fig7


## CELL 15 — Figure 8: PSNR/SSIM Bars + MRI Table


In [20]:
mkeys=[m[0] for m in METHODS]
mshort=['Bicubic','ReLU\nMLP','SIREN\n(A2)','SIREN+\nFFL+CAOS']

fig,axes=plt.subplots(2,2,figsize=(16,12))
for row,scale in enumerate(SCALES_MRI):
    pm=[np.mean([d['psnr'] for d in mri_res[scale][mk]]) for mk in mkeys]
    ps=[np.std ([d['psnr'] for d in mri_res[scale][mk]]) for mk in mkeys]
    sm=[np.mean([d['ssim'] for d in mri_res[scale][mk]]) for mk in mkeys]
    ss=[np.std ([d['ssim'] for d in mri_res[scale][mk]]) for mk in mkeys]

    ax=axes[row,0]
    bars=ax.bar(mshort,pm,yerr=ps,color=MC,edgecolor='black',lw=0.8,capsize=6)
    for b,v in zip(bars,pm): ax.text(b.get_x()+b.get_width()/2,b.get_height()+0.3,f'{v:.2f}',
                                      ha='center',va='bottom',fontsize=11,fontweight='bold')
    ax.set_ylabel('PSNR (dB)',fontsize=13)
    ax.set_title(f'PSNR — {scale}× SR',fontsize=13,fontweight='bold'); ax.grid(True,alpha=0.3,axis='y')

    ax2=axes[row,1]
    bars2=ax2.bar(mshort,sm,yerr=ss,color=MC,edgecolor='black',lw=0.8,capsize=6)
    for b,v in zip(bars2,sm): ax2.text(b.get_x()+b.get_width()/2,b.get_height()+0.005,f'{v:.4f}',
                                        ha='center',va='bottom',fontsize=11,fontweight='bold')
    ax2.set_ylabel('SSIM',fontsize=13); ax2.set_ylim(0,1.09)
    ax2.set_title(f'SSIM — {scale}× SR',fontsize=13,fontweight='bold'); ax2.grid(True,alpha=0.3,axis='y')

plt.suptitle('FIGURE 8 — MRI SR: PSNR & SSIM (Mean ± Std, 5 slices)',
             fontsize=14,fontweight='bold',y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_ROOT,'fig8_mri_psnr_ssim.png'),dpi=150,bbox_inches='tight')
plt.show(); print('Saved fig8')

print('\n'+'-'*75)
print(f'{"Table 3 — MRI Super-Resolution Results (Bonus)":^75}')
print('-'*75)
for scale in SCALES_MRI:
    print(f'\n  {scale}× Upsampling')
    print(f'  {"Method":<28} {"Mean PSNR":>12} {"Mean SSIM":>12}')
    print(f'  {"-"*55}')
    for mk,mname,_,_ in METHODS:
        p=np.mean([d['psnr'] for d in mri_res[scale][mk]])
        s=np.mean([d['ssim'] for d in mri_res[scale][mk]])
        print(f'  {mname:<28} {p:>11.2f} {s:>12.4f}')
print('-'*75)

Saved fig8

---------------------------------------------------------------------------
              Table 3 — MRI Super-Resolution Results (Bonus)               
---------------------------------------------------------------------------

  2× Upsampling
  Method                          Mean PSNR    Mean SSIM
  -------------------------------------------------------
  Bicubic                            26.99       0.8829
  ReLU MLP                           19.31       0.3721
  SIREN (A2)                         26.33       0.8645
  SIREN+FFL+CAOS                     14.20       0.1013

  4× Upsampling
  Method                          Mean PSNR    Mean SSIM
  -------------------------------------------------------
  Bicubic                            22.29       0.6245
  ReLU MLP                           19.05       0.3265
  SIREN (A2)                         21.40       0.5486
  SIREN+FFL+CAOS                     13.33       0.0555
------------------------------------------------

---
## CELL 16 — Save CSVs + Final Checklist


In [21]:
with open(os.path.join(DRIVE_ROOT,'results_helmholtz.csv'),'w',newline='') as f:
    w=csv.writer(f)
    w.writerow(['k','init_loss','final_loss','reduction_pct','steps','time_min','tag'])
    w.writerow([1,A2_H_INIT,A2_H_FINAL,96.9,5000,63,'A2_baseline'])
    for k in K_VALUES:
        r=helm[k]
        w.writerow([k,f'{r["init"]:.0f}',f'{r["final"]:.0f}',f'{r["reduction"]:.1f}',
                    H_STEPS,f'{r["time_min"]:.1f}','A3_repro' if k==1 else 'A3_new'])

with open(os.path.join(DRIVE_ROOT,'results_wave.csv'),'w',newline='') as f:
    w=csv.writer(f)
    w.writerow(['ic_type','init_loss','final_loss','reduction_pct','steps','time_min'])
    w.writerow(['A2_default',A2_W_INIT,A2_W_FINAL,99.98,5000,67])
    for ic in IC_TYPES:
        r=wave[ic]
        w.writerow([ic,f'{r["init"]:.0f}',f'{r["final"]:.0f}',f'{r["reduction"]:.1f}',W_STEPS,f'{r["time_min"]:.1f}'])

with open(os.path.join(DRIVE_ROOT,'results_mri.csv'),'w',newline='') as f:
    w=csv.writer(f)
    w.writerow(['scale','method','slice_idx','psnr','ssim'])
    for sc in SCALES_MRI:
        for mk,_,_,_ in METHODS:
            for i,d in enumerate(mri_res[sc][mk]):
                w.writerow([f'{sc}x',mk,i,f'{d["psnr"]:.4f}',f'{d["ssim"]:.6f}'])

FIGS=['fig1_helmholtz_loss.png','fig2_helmholtz_wavefields.png','fig3_helmholtz_convergence.png',
      'fig4_wave_loss.png','fig5_wave_spacetime.png','fig6_wave_ic_profiles.png',
      'fig7_mri_visual.png','fig8_mri_psnr_ssim.png']

print('='*70)
print('  LAIBA NOOR (25I-8035) — A3 COMPLETE')
print('  Group ID09 | FAST-NUCES | Dr. Zohair Ahmed | April 2026')
print('='*70)
print('\nFigures:')
for fg in FIGS:
    ok='✓' if os.path.exists(os.path.join(DRIVE_ROOT,fg)) else '✗ MISSING'
    print(f'  {ok}  {fg}')
print('\nCSVs: results_helmholtz.csv | results_wave.csv | results_mri.csv')
print(f'\nAll in: {DRIVE_ROOT}')
print('='*70)

  LAIBA NOOR (25I-8035) — A3 COMPLETE
  Group ID09 | FAST-NUCES | Dr. Zohair Ahmed | April 2026

Figures:
  ✓  fig1_helmholtz_loss.png
  ✓  fig2_helmholtz_wavefields.png
  ✓  fig3_helmholtz_convergence.png
  ✓  fig4_wave_loss.png
  ✓  fig5_wave_spacetime.png
  ✓  fig6_wave_ic_profiles.png
  ✓  fig7_mri_visual.png
  ✓  fig8_mri_psnr_ssim.png

CSVs: results_helmholtz.csv | results_wave.csv | results_mri.csv

All in: /content/drive/MyDrive/SIREN_A3_Laiba
